In [1]:
from google.colab import auth
auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from google.colab import drive

drive.mount('/content/gdrive')

df = pd.read_csv('/content/gdrive/My Drive/NCKH/NGHIÊN CỨU KHOA HỌC/Code/P2P_Processed.csv')
print(df.head())

Mounted at /content/gdrive
   acc_now_delinq  acc_open_past_24mths  annual_inc  badloan  \
0               0                     2     49800.0        0   
1               0                     3     34000.0        0   
2               0                     0     34000.0        0   
3               0                     2     50000.0        1   
4               0                     2     29000.0        1   

   chargeoff_within_12_mths  collections_12_mths_ex_med  delinq_2yrs  \
0                         0                           0            0   
1                         0                           0            0   
2                         0                           0            0   
3                         0                           0            0   
4                         0                           0            0   

   delinq_amnt        dti  earnings  ...  purpose_home_improvement  \
0            0  25.120001    995.80  ...                     False   
1            0 

In [2]:
boolean_cols = df.select_dtypes(include=['bool']).columns
df[boolean_cols] = df[boolean_cols].astype(int)

In [3]:
df.columns

Index(['acc_now_delinq', 'acc_open_past_24mths', 'annual_inc', 'badloan',
       'chargeoff_within_12_mths', 'collections_12_mths_ex_med', 'delinq_2yrs',
       'delinq_amnt', 'dti', 'earnings', 'emp_length', 'funded_amnt', 'grade',
       'inq_last_12m', 'inq_last_6mths', 'installment', 'int_rate',
       'loan_amnt', 'loan_status', 'loan_status_5', 'loan_sum', 'loan_vol6m',
       'mort_acc', 'mths_since_last_delinq', 'num_accts_ever_120_pd',
       'num_actv_rev_tl', 'num_tl_30dpd', 'open_acc', 'pct_tl_nvr_dlq',
       'pub_rec', 'pub_rec_bankruptcies', 'pymnt_plan', 'revol_util',
       'sub_grade', 'term', 'issue_year', 'home_ownership_MORTGAGE',
       'home_ownership_NONE', 'home_ownership_OTHER', 'home_ownership_OWN',
       'home_ownership_RENT', 'default_binary', 'purpose_credit_card',
       'purpose_debt_consolidation', 'purpose_home_improvement',
       'purpose_house', 'purpose_major_purchase', 'purpose_medical',
       'purpose_moving', 'purpose_other', 'purpose_renewabl

In [4]:
# Danh sách cột cần loại bỏ (không loại bỏ issue_year ở bước này)
drop_cols = [
    'loan_status', 'badloan', 'pymnt_plan', 'sub_grade', 'loan_sum', 'loan_status_5', 'funded_amnt'
]

# Loại bỏ các cột không mong muốn (trừ issue_year vì cần để tách dữ liệu)
df1 = df.drop(columns=[col for col in drop_cols if col in df.columns], errors='ignore')

# Tách tập huấn luyện (2012 - 2017) và kiểm tra (2018 - 2019)
train_df = df1[df1["issue_year"].between(2012, 2017)].copy()
test_df = df1[df1["issue_year"].between(2018, 2019)].copy()

# Bây giờ có thể loại bỏ issue_year
train_df = train_df.drop(columns=["issue_year"], errors="ignore")
test_df = test_df.drop(columns=["issue_year"], errors="ignore")

# Chia X và y
y_train = train_df["default_binary"]
X_train = train_df.drop(columns=["default_binary"], errors="ignore")

y_test = test_df["default_binary"]
X_test = test_df.drop(columns=["default_binary"], errors="ignore")

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import StackingClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.pipeline import make_pipeline
from lightgbm import LGBMClassifier
from sklearn.ensemble import GradientBoostingClassifier

# Stacking: LR - RF - XGBoost

In [7]:
# Áp dụng SMOTE
smote = SMOTE(sampling_strategy=0.5, random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

# Chuẩn hóa dữ liệu
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_balanced)
X_test_scaled = scaler.transform(X_test)

# Mô hình con
logit_model = LogisticRegression(C=0.01, penalty='l1', solver='liblinear', max_iter=500)
rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, min_samples_split=20, min_samples_leaf=10, random_state=42)
xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, subsample=0.7, colsample_bytree=0.7, random_state=42)

# Sử dụng StackingClassifier
stacking_model = StackingClassifier(
    estimators=[
        ('lr', logit_model),
        ('rf', rf_model)
    ],
    final_estimator=xgb_model,
    passthrough=True
)

# Huấn luyện mô hình Stacking
stacking_model.fit(X_train_scaled, y_train_balanced)

# Dự đoán
final_preds = stacking_model.predict(X_test_scaled)

# Đánh giá
accuracy = accuracy_score(y_test, final_preds)
roc_auc = roc_auc_score(y_test, final_preds)

print(f"Accuracy: {accuracy:.4f}")
print(f"ROC AUC Score: {roc_auc:.4f}")


KeyboardInterrupt: 

# Boosting

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

drop_cols = ['loan_status', 'badloan', 'pymnt_plan', 'sub_grade', 'loan_status_5', 'funded_amnt', 'issue_year']
df1 = df.drop(columns=[col for col in drop_cols if col in df.columns], errors='ignore')

# Chia X và y
y = df1['default_binary']
X = df1.drop(columns=['default_binary'])

# Cân bằng dữ liệu bằng SMOTE
smote = SMOTE(sampling_strategy=0.5, random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Chia tập train/test
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

# Chuẩn hóa dữ liệu
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import GradientBoostingClassifier

# XGBoost
xgb_model = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=42)
xgb_model.fit(X_train_scaled, y_train)
xgb_preds = xgb_model.predict(X_test_scaled)

# LightGBM
lgbm_model = LGBMClassifier(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42)
lgbm_model.fit(X_train_scaled, y_train)
lgbm_preds = lgbm_model.predict(X_test_scaled)

# Gradient Boosting
gbm_model = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42)
gbm_model.fit(X_train_scaled, y_train)
gbm_preds = gbm_model.predict(X_test_scaled)

# Đánh giá mô hình
xgb_acc = accuracy_score(y_test, xgb_preds)
xgb_auc = roc_auc_score(y_test, xgb_preds)

lgbm_acc = accuracy_score(y_test, lgbm_preds)
lgbm_auc = roc_auc_score(y_test, lgbm_preds)

gbm_acc = accuracy_score(y_test, gbm_preds)
gbm_auc = roc_auc_score(y_test, gbm_preds)

print(f"XGBoost - Accuracy: {xgb_acc:.4f}, ROC AUC: {xgb_auc:.4f}")
print(f"LightGBM - Accuracy: {lgbm_acc:.4f}, ROC AUC: {lgbm_auc:.4f}")
print(f"Gradient Boosting - Accuracy: {gbm_acc:.4f}, ROC AUC: {gbm_auc:.4f}")

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Info] Number of positive: 987974, number of negative: 1976257
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.077584 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3645
[LightGBM] [Info] Number of data points in the train set: 2964231, number of used features: 45
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.333299 -> initscore=-0.693304
[LightGBM] [Info] Start training from score -0.693304


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


# Hybrid

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

drop_cols = ['loan_status', 'badloan', 'pymnt_plan', 'sub_grade', 'loan_status_5', 'funded_amnt']
df1 = df.drop(columns=[col for col in drop_cols if col in df.columns], errors='ignore')

# Chia X và y
y = df1['default_binary']
X = df1.drop(columns=['default_binary'])

# Cân bằng dữ liệu bằng SMOTE
smote = SMOTE(sampling_strategy=0.5, random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Chia tập train/test
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

# Chuẩn hóa dữ liệu
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# Random Forest giai đoạn 1
rf_model = RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42)
rf_model.fit(X_train_scaled, y_train)

# Lấy xác suất dự đoán từ Random Forest
rf_train_preds = rf_model.predict_proba(X_train_scaled)[:, 1].reshape(-1, 1)
rf_test_preds = rf_model.predict_proba(X_test_scaled)[:, 1].reshape(-1, 1)

# XGBoost giai đoạn 2
xgb_hybrid_model = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42)
xgb_hybrid_model.fit(rf_train_preds, y_train)

# Dự đoán và đánh giá
hybrid_preds = xgb_hybrid_model.predict(rf_test_preds)
hybrid_acc = accuracy_score(y_test, hybrid_preds)
hybrid_auc = roc_auc_score(y_test, hybrid_preds)

print(f"Hybrid Model - Accuracy: {hybrid_acc:.4f}, ROC AUC: {hybrid_auc:.4f}")